# Stage 2 and Stage 3 Run Commands

Current workflow for this repo:

- SKU-110K data is in `data/SKU110K`.
- Product detector checkpoint is `s3_iou_resnet50_csv_06.h5`.
- Stage 2 writes `data/processed/s3_detections.json`.
- Stage 3 combines `row_predictions.csv` and `detections.json`.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/Users/dipali/Documents/Coursework/ACV/Final_projectt/CS543_row_detection')
os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

## 0. Check Required Files

In [ ]:
paths = [
    Path('s3_iou_resnet50_csv_06.h5'),
    Path('data/SKU110K/images'),
    Path('data/SKU110K/annotations/annotations_train.csv'),
    Path('data/SKU110K/annotations/annotations_val.csv'),
    Path('data/processed/SHARD/images'),
    Path('data/processed/s3_row_predictions.csv'),
]

for path in paths:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

## 1. Environment For Provided H5 Detector

`s3_iou_resnet50_csv_06.h5` is an old Keras RetinaNet model. It usually requires a Python 3.10 environment with `keras_retinanet`. Python 3.13/Keras 3 may fail.

Run these commands in a terminal if this notebook kernel cannot import `keras_retinanet`.

In [ ]:
# Terminal commands:
# conda create -n sku-retinanet python=3.10 -y
# conda activate sku-retinanet
# python -m pip install tensorflow==2.13.1 keras==2.13.1 opencv-python pillow tqdm h5py matplotlib
# python pip install tensorflow-metal
# python -m pip install git+https://github.com/fizyr/keras-retinanet.git

In [ ]:
import importlib.util
print('keras_retinanet installed:', importlib.util.find_spec('keras_retinanet') is not None)

In [ ]:
import sys
print(sys.executable)

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices())

## 2. Stage 1: Export Shelf Row Predictions

Skipping if `data/processed/s3_row_predictions.csv` already exists.

In [ ]:
!python scripts/s3_export_row_predictions.py \
  --checkpoint runs/row_dht_1d/s3_best.pt \
  --images_dir data/processed/SHARD/images \
  --output data/processed/s3_row_predictions.csv \
  --threshold 0.35 \
  --min_distance 20

## 3. Stage 2: Product Detection With Provided H5 Model

In [ ]:
!python scripts/s3_predict_product_detections_h5.py \
  --model s3_iou_resnet50_csv_06.h5 \
  --images_dir data/processed/SHARD/images \
  --output data/processed/s3_detections.json \
  --score_threshold 0.5 \
  --max_detections 300

## 4. Visualize Stage 2 Detections

In [ ]:
!python scripts/s3_visualize_detections.py \
  --detections data/processed/s3_detections.json \
  --image_dir data/processed/SHARD/images \
  --output_dir runs/product_detection_visualizations \
  --limit 50

## 5. Stage 3: Product Localization

In [ ]:
!python scripts/s3_localize.py \
  --row_preds data/processed/s3_row_predictions.csv \
  --detections data/processed/s3_detections.json \
  --image_dir data/processed/SHARD/images \
  --output data/processed/s3_localized_products.json \
  --output_csv data/processed/s3_localized_products.csv

## 6. Visualize Stage 3 Localization

In [ ]:
!python scripts/s3_visualize_localization.py \
  --localized data/processed/s3_localized_products.json \
  --image_dir data/processed/SHARD/images \
  --row_preds data/processed/s3_row_predictions.csv \
  --output_dir runs/localization_visualizations \
  --limit 50

## 7. Inspect Outputs

In [ ]:
import json

outputs = [
    Path('data/processed/s3_detections.json'),
    Path('data/processed/s3_localized_products.json'),
    Path('data/processed/s3_localized_products.csv'),
    Path('runs/product_detection_visualizations'),
    Path('runs/localization_visualizations'),
]

for path in outputs:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

det_path = Path('data/processed/s3_detections.json')
if det_path.exists():
    detections = json.loads(det_path.read_text())
    print('detections:', len(detections))
    print('sample:', detections[0] if detections else None)